In [6]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"

In [7]:
import json

# Load Harsh's cleaned ABSA output
with open("models/absa/task3_cleaned_aspects.json", "r") as f:
    data = json.load(f)

print("Data loaded successfully")
print("Total reviews:", len(data))

Data loaded successfully
Total reviews: 5000


In [8]:
all_aspects = []

for review_id, aspects in data.items():
    for item in aspects:
        aspect = item["aspect"].lower().strip()
        all_aspects.append(aspect)

print("Total aspect mentions:", len(all_aspects))

Total aspect mentions: 76172


In [9]:
unique_aspects = list(set(all_aspects))

print("Unique aspects:", len(unique_aspects))
print("\nSample aspects:")
print(unique_aspects[:30])

Unique aspects: 27860

Sample aspects:
['4lb box', 'bitterness', 'name', '2 3 ounces', '240 packets', 'wonder sweetener', 'just a little drop', 'mothers', 'corn chips', 'ham flavor', 'bright steel bottom', 'dark golden brown', 'current favorite coffee vessel', 'two sticks', "our farmers' cooperatives", 'slightly stale tasting', 'no legitimate benefit', 'different bar', 'pc entrees', 'authentic british tea flavor', 'gold box', 'fine quenching', 'last cup', 'bread machine', 'drawbacks', 'i finaly', 'your own personal preference', 'noses', 'large hotel suppliers', 'not so smooth items']


In [10]:
def strong_filter(a):
    a = a.lower().strip()
    
    # remove non-letters
    import re
    a = re.sub(r'[^a-zA-Z\s]', '', a)
    
    words = a.split()
    
    # keep ONLY 1–2 word phrases
    if len(words) > 2:
        return None
    
    # remove very short words
    if len(a) < 3:
        return None
    
    return a

In [11]:
cleaned_aspects = []

for a in unique_aspects:
    a_clean = strong_filter(a)
    if a_clean:
        cleaned_aspects.append(a_clean)

cleaned_aspects = list(set(cleaned_aspects))

print("After strong filter:", len(cleaned_aspects))
print(cleaned_aspects[:30])

After strong filter: 17334
['whole nuts', 'recommended settings', 'bitterness', 'french press', 'macaroni', 'replacment set', 'perimeter', ' mg sodium', 'stinky powder', 'new foods', 'terrible shipping', 'name', 'only cups', 'wonder sweetener', 'mothers', 'corn chips', 'breathe', 'one exception', 'only treat', 'rash', 'ham flavor', 'green onion', 'backpack', 'first  pounds', 'positive thing', 'our household', 'ghee taste', 'cola drinker', 'guarana', 'slaughterhouse leftovers']


In [12]:
bad_words = {
    "thing", "product", "item", "stuff", "something",
    "anything", "everything", "nothing",
    "time", "day", "way", "lot", "kind",
    "same", "many", "little", "great"
}

filtered_aspects = [
    a for a in cleaned_aspects
    if a not in bad_words
]

print("After removing generic:", len(filtered_aspects))
print(filtered_aspects[:30])

After removing generic: 17333
['whole nuts', 'recommended settings', 'bitterness', 'french press', 'macaroni', 'replacment set', 'perimeter', ' mg sodium', 'stinky powder', 'new foods', 'terrible shipping', 'name', 'only cups', 'wonder sweetener', 'mothers', 'corn chips', 'breathe', 'one exception', 'only treat', 'rash', 'ham flavor', 'green onion', 'backpack', 'first  pounds', 'positive thing', 'our household', 'ghee taste', 'cola drinker', 'guarana', 'slaughterhouse leftovers']


In [13]:
from collections import Counter

aspect_counts = Counter(all_aspects)

filtered_aspects = [
    a for a in filtered_aspects
    if aspect_counts[a] >= 10   # increase threshold
]

print("Final aspect count:", len(filtered_aspects))

Final aspect count: 717


In [14]:
import subprocess
subprocess.run(["pip", "install", "-U", "sentence-transformers", "huggingface_hub"], 
               capture_output=True)

CompletedProcess(args=['pip', 'install', '-U', 'sentence-transformers', 'huggingface_hub'], returncode=0, stdout=b'Requirement already satisfied: sentence-transformers in /Users/kanishkagupta/main-env/lib/python3.9/site-packages (5.1.2)\nRequirement already satisfied: huggingface_hub in /Users/kanishkagupta/main-env/lib/python3.9/site-packages (0.36.2)\nCollecting huggingface_hub\n  Using cached huggingface_hub-1.8.0-py3-none-any.whl.metadata (13 kB)\nRequirement already satisfied: transformers<5.0.0,>=4.41.0 in /Users/kanishkagupta/main-env/lib/python3.9/site-packages (from sentence-transformers) (4.57.6)\nRequirement already satisfied: tqdm in /Users/kanishkagupta/main-env/lib/python3.9/site-packages (from sentence-transformers) (4.67.1)\nRequirement already satisfied: torch>=1.11.0 in /Users/kanishkagupta/main-env/lib/python3.9/site-packages (from sentence-transformers) (2.8.0)\nRequirement already satisfied: scikit-learn in /Users/kanishkagupta/main-env/lib/python3.9/site-packages 

In [15]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully")

/Users/kanishkagupta/main-env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Model loaded successfully


In [23]:
import re
import json
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

# ── 1. Load data ──────────────────────────────────────────────────────────────
with open("models/absa/task3_cleaned_aspects.json", "r") as f:
    data = json.load(f)

df = pd.read_csv("/Users/kanishkagupta/Downloads/SEM 2_SPRING2026/IS567_TM/project/Sentiment-Intelligence-Platform/Product Reviews.csv")
df = df.reset_index(drop=True)
index_to_product = {str(i): row["ProductId"] for i, row in df.iterrows()}

# ── 2. Clean aspects ──────────────────────────────────────────────────────────
def strong_filter(a):
    a = a.lower().strip()
    a = re.sub(r'[^a-zA-Z\s]', '', a)
    a = a.strip()
    words = a.split()
    if len(words) > 2 or len(a) < 3:
        return None
    return a

bad_words = {
    "thing", "product", "item", "stuff", "something",
    "anything", "everything", "nothing", "time", "day",
    "way", "lot", "kind", "same", "many", "little", "great"
}

cleaned_mention_counts = Counter()
review_aspect_map = {}

for review_id, aspects in data.items():
    cleaned_pairs = []
    for item in aspects:
        raw = item["aspect"].lower().strip()
        cleaned = strong_filter(raw)
        if cleaned and cleaned not in bad_words:
            cleaned_mention_counts[cleaned] += 1
            cleaned_pairs.append((cleaned, item["sentiment"]))
    review_aspect_map[review_id] = cleaned_pairs

final_aspects = [a for a, c in cleaned_mention_counts.items() if c >= 10]
final_aspect_set = set(final_aspects)
print(f"✅ Aspects: {len(final_aspect_set)}")

# ── 3. Embed + cluster ────────────────────────────────────────────────────────
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(final_aspects, show_progress_bar=True)
embeddings = normalize(embeddings)

N_CLUSTERS = 25
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)

cluster_df = pd.DataFrame({"aspect": final_aspects, "cluster": cluster_labels})
print("✅ Clustering done")

# ── 4. Taxonomy ───────────────────────────────────────────────────────────────
cluster_to_taxonomy = {
    0:  "food_ingredients",
    1:  "noise",
    2:  "noise",
    3:  "beverage",
    4:  "noise",
    5:  "taste_flavor",
    6:  "packaging_format",
    7:  "noise",
    8:  "noise",
    9:  "availability_location",
    10: "packaging_delivery",
    11: "noise",
    12: "noise",
    13: "noise",
    14: "noise",
    15: "freshness_aroma",
    16: "food_type",
    17: "brand_quality",
    18: "noise",
    19: "price_value",
    20: "noise",
    21: "quantity_size",
    22: "noise",
    23: "nutrition_ingredients",
    24: "target_user",
}

VALID_TAXONOMY = {
    "food_ingredients", "beverage", "taste_flavor", "packaging_format",
    "availability_location", "packaging_delivery", "freshness_aroma",
    "food_type", "brand_quality", "price_value", "quantity_size",
    "nutrition_ingredients", "target_user"
}

aspect_to_taxonomy = {
    row["aspect"]: cluster_to_taxonomy[row["cluster"]]
    for _, row in cluster_df.iterrows()
}

sentiment_map = {"positive": 1, "neutral": 0, "negative": -1}
taxonomy_labels = sorted(VALID_TAXONOMY)
taxonomy_index = {label: i for i, label in enumerate(taxonomy_labels)}
print("✅ Taxonomy built")

# ── 5. Build product vectors ──────────────────────────────────────────────────
product_vectors = defaultdict(lambda: defaultdict(list))

for review_id, pairs in review_aspect_map.items():
    product_id = index_to_product.get(str(review_id), str(review_id))
    for aspect, sentiment in pairs:
        if aspect in final_aspect_set:
            norm = aspect_to_taxonomy.get(aspect)
            if norm and norm in VALID_TAXONOMY:
                score = sentiment_map.get(sentiment.lower(), 0)
                product_vectors[product_id][norm].append(score)

records = []
for product_id, aspect_scores in product_vectors.items():
    vec = np.zeros(len(taxonomy_labels))
    for aspect_name, scores in aspect_scores.items():
        idx = taxonomy_index[aspect_name]
        vec[idx] = np.mean(scores)
    records.append({"product_id": product_id, **dict(zip(taxonomy_labels, vec))})

product_vector_df = pd.DataFrame(records).fillna(0)
print(f"✅ Product vectors shape: {product_vector_df.shape}")
print(product_vector_df.head())

# ── 6. Save ───────────────────────────────────────────────────────────────────
product_vector_df.to_csv("models/absa/product_aspect_vectors.csv", index=False)
with open("models/absa/aspect_taxonomy_map.json", "w") as f:
    json.dump(aspect_to_taxonomy, f, indent=2)

print("\n✅ Stage 3 Complete!")
print("Saved: product_aspect_vectors.csv")
print("Saved: aspect_taxonomy_map.json")

✅ Aspects: 747


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

✅ Clustering done
✅ Taxonomy built
✅ Product vectors shape: (691, 14)
   product_id  availability_location  beverage  brand_quality  \
0  B001E4KFG0                    1.0       1.0            0.0   
1  B000LQOCH0                    0.0       0.0            0.0   
2  B006K2ZZ7K                    1.0       0.0            0.0   
3  B000E7L2R4                   -1.0       0.0            0.0   
4  B00171APVA                    1.0       0.0            0.0   

   food_ingredients  food_type  freshness_aroma  nutrition_ingredients  \
0               0.0        0.0              1.0                    0.0   
1               0.0       -1.0              0.0                    0.0   
2               0.0        0.0              1.0                    0.0   
3               0.0        0.0              0.0                    1.0   
4               0.0        0.0              1.0                    0.0   

   packaging_delivery  packaging_format  price_value  quantity_size  \
0                 0.0  